In [1]:
import os
model_to_test = 'jet_baseline'
model_revision = 'the_baseline'
hls4ml_revision = 'VitisUnified'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, str(model_revision))
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration

- **Model Name**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Backend**: VitisUnified
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Architecture**: Functional model with 3 dense layers (64→32→32) + output layer
- **Dataset**: HLS4ML LHC Jets
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [2]:
import os
import keras
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

2026-05-06 02:17:46.602980: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778026666.661946  202081 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778026666.681642  202081 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 02:17:46.798586: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
X_train_val = np.load('Data/x_train_val.npy')
X_test = np.load('Data/x_test.npy')
y_train_val = np.load('Data/y_train_val.npy')
y_test = np.load('Data/y_test.npy')
classes = np.load('Data/classes.npy', allow_pickle=True)

In [8]:

inputs = keras.layers.Input(shape=(16,), name='input_layer')
x = keras.layers.Dense(64, activation='relu', name='dense_0')(inputs)
x = keras.layers.Dense(32, activation='relu', name='dense_1')(x)
x = keras.layers.Dense(32, activation='relu', name='dense_2')(x)
outputs = keras.layers.Dense(5, activation='softmax', name='dense_3')(x)

model = keras.Model(inputs=inputs, outputs=outputs)

opt = keras.optimizers.Adam(learning_rate=5e-3)

model.compile(opt, loss=['categorical_crossentropy'], metrics=['accuracy'])

In [9]:
model.summary()

# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_0 (Dense)                 │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,389 (17.14 KB)

 Trainable params: 4,389 (17.14 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
import os
from keras.callbacks import EarlyStopping, ReduceLROnPlateau


callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=6, verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=2, min_lr=1e-6, verbose=1),
]

In [11]:
model.fit(
    X_train_val, y_train_val,
    validation_data=(X_test, y_test),
    epochs=60,
    batch_size=1024,
    callbacks=callbacks,
    shuffle=True
)

Epoch 1/60


2026-05-06 02:20:38.135861: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_464', 124 bytes spill stores, 120 bytes spill loads

2026-05-06 02:20:38.475865: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_464', 2900 bytes spill stores, 2908 bytes spill loads

2026-05-06 02:20:38.574000: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_464', 4368 bytes spill stores, 4372 bytes spill loads



649/649 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.7348 - loss: 0.7473 - val_accuracy: 0.7525 - val_loss: 0.6929 - learning_rate: 0.0050
Epoch 2/60
649/649 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7594 - loss: 0.6739 - val_accuracy: 0.7587 - val_loss: 0.6780 - learning_rate: 0.0050
Epoch 3/60
649/649 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7624 - loss: 0.6643 - val_accuracy: 0.7604 - val_loss: 0.6696 - learning_rate: 0.0050
Epoch 4/60
649/649 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7639 - loss: 0.6596 - val_accuracy: 0.7629 - val_loss: 0.6624 - learning_rate: 0.0050
Epoch 5/60
649/649 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7651 - loss: 0.6563 - val_accuracy: 0.7638 - val_loss: 0.6592 - learning_rate: 0.0050
Epoch 6/60
649/649 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7655 - loss: 0.6545 - val_accuracy: 0.7633 - val_loss: 0.6601 - learning_rate: 0.0050
Epoch 7/60
642/649 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7659 - loss: 0.6525
Epoch 7: Re

In [13]:
score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])
test_acc = score[1]

model_name=f"model_{model_revision}_acc={test_acc:.4f}.keras"
model.save(os.path.join(model_dir, model_name))

Test loss: 0.641478955745697
Test accuracy: 0.7685180902481079
